# Constraint Satisfaction Problem

## Aufgabe: 
### Labor- und Präsentationsplanung als Constraint Satisfaction Problem (CSP)


In [5]:
%pip install python-constraint pandas

/Users/heofthetea/Documents/Development/dhbw/semester-6/grundlagen_ki/Constraint-AI/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


### CSP-Konfiguration laden und darstellen

Tragen Sie den Namen des Konfigurationsfiles ein. Dieses wird kurz analysiert und dargestellt. Danach können Sie es geeignet dem CSP übergeben.


In [ ]:
# Ihr Dateiname - nacheinander mehrere Konfigurationen testen und analysieren.
from csp_utils import analyze_and_display
config_file = "configB.json"
analyze_and_display(config_file)


# CSP-Konfiguration

Verwendete Datei: /Users/heofthetea/Documents/Development/dhbw/semester-6/grundlagen_ki/Constraint-AI/C1/configC.json
Lademodus: json


## Übersicht

Gruppen:
  G1, G2, G3, G4, G5, G6, G7, G8, G9
Tage:
  Mon, Tue, Wed, Thu, Fri
Zeitslots:
  1: 08:00–10:00
  2: 10:30–12:30
  3: 13:00–14:30
  4: 15:00–17:00
Räume:
  L1, L2, L3
Anzahl Kommissionen:
  3
Anzahl Verfügbarkeits-Einträge:
  45


## Kompetenzen der Kommissionen

...

## Legende

Kommission,Farbe
K1,
K2,
K3,


## Wochenplan der Verfügbarkeiten

,Mon,Tue,Wed,Thu,Fri
Zeitslot,,,,,
1 (08:00–10:00),K3,"K2, K3","K1, K2, K3","K1, K2, K3","K1, K2"
2 (10:30–12:30),K3,"K1, K2, K3","K1, K2, K3","K1, K2, K3","K1, K2"
3 (13:00–14:30),K3,"K1, K2, K3","K1, K2, K3","K2, K3","K1, K2"
4 (15:00–17:00),K3,"K1, K2, K3","K1, K2, K3","K2, K3","K1, K2"


{'groups': ['G1', 'G2', 'G3', 'G4', 'G5', 'G6', 'G7', 'G8', 'G9'],
 'days': ['Mon', 'Tue', 'Wed', 'Thu', 'Fri'],
 'timeslots': {'1': '08:00–10:00',
  '2': '10:30–12:30',
  '3': '13:00–14:30',
  '4': '15:00–17:00'},
 'rooms': ['L1', 'L2', 'L3'],
 'commissions': {'K1': ['A', 'B'], 'K2': ['A'], 'K3': ['A', 'C']},
 'availability': {'K1': [['Tue', 2],
   ['Tue', 3],
   ['Tue', 4],
   ['Wed', 1],
   ['Wed', 2],
   ['Wed', 3],
   ['Wed', 4],
   ['Thu', 1],
   ['Thu', 2],
   ['Fri', 1],
   ['Fri', 2],
   ['Fri', 3],
   ['Fri', 4]],
  'K2': [['Tue', 1],
   ['Tue', 2],
   ['Tue', 3],
   ['Tue', 4],
   ['Wed', 1],
   ['Wed', 2],
   ['Wed', 3],
   ['Wed', 4],
   ['Thu', 1],
   ['Thu', 2],
   ['Thu', 3],
   ['Thu', 4],
   ['Fri', 1],
   ['Fri', 2],
   ['Fri', 3],
   ['Fri', 4]],
  'K3': [['Mon', 1],
   ['Mon', 2],
   ['Mon', 3],
   ['Mon', 4],
   ['Tue', 1],
   ['Tue', 2],
   ['Tue', 3],
   ['Tue', 4],
   ['Wed', 1],
   ['Wed', 2],
   ['Wed', 3],
   ['Wed', 4],
   ['Thu', 1],
   ['Thu', 2],
   ['Th

## Definition des Constraint Problems

Sie können die Bibliothek aus der Vorlesung nutzen, aber auch andere, z.B. diese:

| Bibliothek            | Vorteile |
|----------------------|---------|
| python-constraint    | Sehr einfach zu nutzen, ideal für klassische CSPs, perfekt für einfache Einführungsaufgaben |
| Google OR-Tools      | Sehr leistungsfähig, unterstützt komplexe Optimierung und Scheduling, industrieller Standard |
| Pyomo                | Saubere mathematische Modellierung, gut für lineare und ganzzahlige Optimierung |
| MiniZinc             | Speziell für Constraint-Probleme entwickelt, klare und strukturierte Modellierung |

In [42]:
from constraint import Problem, AllDifferentConstraint
from itertools import combinations
import json

with open(config_file, "r") as f:
    config = json.load(f)

len_days = len(config["days"])
len_timeslots = len(config["timeslots"])

DAYS_TO_INT = {"Mon": 0, "Tue": 1, "Wed": 2, "Thu": 3, "Fri": 4}


PRESENTATION_ROOMS = {"A": ["L1", "L2"], "B": ["L3"], "C": ["L1", "L2", "L3"]}


# The two hardest things in programming are naming things, cache invalidation and off-by-one errors.
dhbw = Problem()

# domain for the variable: The details for the slot a presentation is held in
slot_domains = [
    (day, slot, commission, room)
    for day in config["days"]
    for slot in config["timeslots"]
    for commission in config["commissions"]
    for room in config["rooms"]
]

# primary variable - pairs of groups and the presentation they are holding
presentations = [
    (group, presentation)
    for group in config["groups"]
    for presentation in ["A", "B", "C"]
]

for pres in presentations:
    dhbw.addVariable(pres, slot_domains)

## hard constraints
for pres in presentations:
    # 7. Präsentationen können nur in bestimmten Räumen gehalten werden
    def matches_room_constraint(topic):
        return lambda slot: slot[3] in PRESENTATION_ROOMS[topic]
    
    dhbw.addConstraint(
        matches_room_constraint(pres[1]),
        (pres,)
    )
    
    # 6. Präsentationen können nur von bestimmten Kommissionen betreut werden
    def matches_commission_constraint(topic):
        return lambda slot: topic in config["commissions"][slot[2]]
    
    dhbw.addConstraint(
        matches_commission_constraint(pres[1]),
        (pres,)
    )

    # 8. Kommissionen können nur an bestimmten Slots
    dhbw.addConstraint(
        lambda slot: [slot[0], int(slot[1])] in config["availability"][slot[2]],
        (pres,)
    )
    

for pres_a, pres_b in combinations(presentations, 2):

    # 1. Keine Kommission doppelt belegt
    # 3. Kein Raum doppelt belegt
    dhbw.addConstraint(
        lambda slot_a, slot_b: slot_a[0] != slot_b[0]
        or slot_a[1] != slot_b[1]
        or (slot_a[2] != slot_b[2] and slot_a[3] != slot_b[3]),
        (pres_a, pres_b),
    )
    

    # Constraints affecting only presentations held by the same group
    if pres_a[0] == pres_b[0]:

        # 9. Maximal eine Präsentation pro Tag pro Gruppe
        # (impliziert 2. Keine Projektgruppe zu einem Zeitpunkt doppelt belegt)
        dhbw.addConstraint(
            lambda slot_a, slot_b: slot_a[0] != slot_b[0], (pres_a, pres_b)
        )

        if pres_a[1] < pres_b[1]:
        # 4. Präsentationsreihenfolge A -> B -> C
            dhbw.addConstraint(
                lambda slot_a, slot_b: DAYS_TO_INT[slot_a[0]] < DAYS_TO_INT[slot_b[0]],
                (pres_a, pres_b),
            )

            # 5. Mindestabstand zwischen Präsentationen jeder Gruppe von 2 Zeitslots
            dhbw.addConstraint(
                lambda slot_a, slot_b: (int(slot_b[1]) + 4 * DAYS_TO_INT[slot_b[0]])
                - (int(slot_a[1]) + 4 * DAYS_TO_INT[slot_a[0]])
                > 2,
                (pres_a, pres_b),
            )

for pres_a, pres_b, pres_c, pres_d in combinations(presentations, 4):
    def ensure_lazy_commissions(slot_a, slot_b, slot_c, slot_d) -> bool:
        # Check if all are on the same day
        if pres_a[0] == pres_b[0] and pres_a[0] == pres_c[0] and pres_a[0] == pres_d[0]:
            # print('hellooooo')
            # sort slots by their time slot
            slots_sorted = sorted([slot_a, slot_b, slot_c, slot_d], key=lambda item: (item[1]))
            print(slots_sorted)
            room_switches = 0
            previous_room = slot_a[3]
            for sl in slots_sorted[1:]:
                if sl[3] != previous_room:
                    room_switches += 1
                    previous_room = sl[3]
            return room_switches <= 1
        return True
    
    dhbw.addConstraint(ensure_lazy_commissions, (pres_a, pres_b, pres_c, pres_d))

solution = dhbw.getSolution()
if solution:
    solution = dict(sorted(solution.items(), key=lambda item: (item[0][0], item[0][1])))
solution
# print(solutions)
# print(solutions.keys[0])

KeyboardInterrupt: 

## Ausführen und Ergebnisanalyse

Implementieren. Begründen. Testen. Interpretieren.

In [ ]:
## config A
{
    ("G5", "A"): ("Tue", "4", "K5", "L2"),
    ("G1", "A"): ("Wed", "4", "K5", "L2"),
    ("G3", "A"): ("Wed", "3", "K5", "L2"),
    ("G5", "B"): ("Wed", "4", "K4", "L3"),
    ("G4", "A"): ("Wed", "2", "K5", "L2"),
    ("G2", "A"): ("Wed", "4", "K3", "L1"),
    ("G2", "B"): ("Thu", "3", "K4", "L3"),
    ("G4", "B"): ("Thu", "1", "K4", "L3"),
    ("G3", "B"): ("Thu", "2", "K4", "L3"),
    ("G1", "B"): ("Thu", "4", "K4", "L3"),
    ("G5", "C"): ("Thu", "4", "K5", "L2"),
    ("G1", "C"): ("Fri", "4", "K5", "L2"),
    ("G2", "C"): ("Fri", "3", "K5", "L2"),
    ("G3", "C"): ("Fri", "2", "K5", "L2"),
    ("G4", "C"): ("Fri", "1", "K5", "L2"),
}


## config B
{
    ("G2", "A"): ("Wed", "4", "K1", "L1"),
    ("G1", "A"): ("Wed", "4", "K3", "L2"),
    ("G3", "A"): ("Wed", "3", "K3", "L2"),
    ("G1", "B"): ("Thu", "4", "K2", "L3"),
    ("G2", "B"): ("Thu", "3", "K2", "L3"),
    ("G3", "B"): ("Thu", "2", "K2", "L3"),
    ("G2", "C"): ("Fri", "4", "K1", "L2"),
    ("G1", "C"): ("Fri", "4", "K3", "L3"),
    ("G3", "C"): ("Fri", "3", "K3", "L3"),
}

## Constraint Validator
Implementieren. Prüfen. Begründen und interpretieren.

### Unmöglichkeit von Config C

Konfiguration C besitzt kein Modell, dass alle Constraints erfüllt. Dies lässt sich beweisen anhand der folgenden Gegebenheiten zeigen:
1. Alle Präsentationen müssen in der Reihenfolge A -> B -> C gehalten werden (Constraint)
2. Eine Gruppe kann nur eine Präsentation pro Tag halten (Constraint)
3. Eine Kommission kann nicht zwei Vorträge parallel betreuen (Constraint)
4. Kommission "K1" ist die einzige, die Präsentation B betreuen kann
5. Kommission "K3" ist die einzige, die Präsentation C betreuen kann
6. K3 ist am Freitag nicht verfügbar

Aus (3) und (4) ergibt sich, dass es gibt eine Gruppe `G` gibt, die ihre B-Präsentation im 9. Verfügbarkeits-Slot von Kommission `K1` halten muss. Dieser 9. Slot ist Slot 2 am Donnerstag. Aus (1) und (2) folgt, dass diese Gruppe `G` ihre C-Präsentation am Tag darauf halten muss, also am Freitag. Aus (5) und (6) folgt, dass es am Freitag keinen Slot mehr geben kann, an dem `G` präsentieren kann.

Dadurch ist bewiesen, dass dieser Datensatz keine Lösung besitzt.